In [2]:
import subprocess
import pandas as pd
from pathlib import Path
import polars as pl
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from sklearn.model_selection import StratifiedGroupKFold
import lightgbm as lgb
import joblib

pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',None)

ROOT=Path("/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files")
TRAIN_DIR=ROOT/"train"
TEST_DIR=ROOT/"test"


In [2]:
#read the files number
dirs=[TRAIN_DIR,TEST_DIR]
for dir in dirs:
    print(f"files under {dir}: {len(list(dir.glob('*parquet')))}")


#read the train files size
cmd=f'du {TRAIN_DIR}/* -mh|sort -rh '
res=subprocess.run(cmd,shell=True,stdout=subprocess.PIPE,text=True)
res_=res.stdout.split('\n')
res1=[line.split("\t") for line in res_ if line]
red=pd.DataFrame(res1,columns=["size","path"])
red["filename"]=red["path"].str.replace("train_","").apply(lambda x:Path(x).stem)

#read the test files size
cmd=f'du {TEST_DIR}/* -h|sort -rh '
res_test=subprocess.run(cmd,shell=True,stdout=subprocess.PIPE,text=True)
res_test_=res_test.stdout.split('\n')
res1_test=[line.split("\t") for line in res_test_ if line]
red_test=pd.DataFrame(res1_test,columns=["size","path"])
red_test["filename"]=red_test["path"].str.replace("test_","").apply(lambda x:Path(x).stem)

red_test.merge(red,on="filename",how="left",suffixes=("_test","_train")).sort_values(by="filename")



files under /home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train: 32
files under /home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test: 36


,size_test,path_test,filename,size_train,path_train
12,32K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_applprev_1_0.parquet,applprev_1_0,102M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_applprev_1_0.parquet
11,32K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_applprev_1_1.parquet,applprev_1_1,70M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_applprev_1_1.parquet
10,32K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_applprev_1_2.parquet,applprev_1_2,NaN,NaN
33,8.0K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_applprev_2.parquet,applprev_2,28M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_applprev_2.parquet
35,4.0K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_base.parquet,base,6.8M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_base.parquet
7,60K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_0.parquet,credit_bureau_a_1_0,62M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_0.parquet
6,60K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_1.parquet,credit_bureau_a_1_1,175M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_1.parquet
5,60K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_2.parquet,credit_bureau_a_1_2,118M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_2.parquet
4,60K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_3.parquet,credit_bureau_a_1_3,69M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_3.parquet
3,60K,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_4.parquet,credit_bureau_a_1_4,NaN,NaN


# READ TRAIN BASE FILE

In [ ]:
#we use pandas dataframe as the base file is not that large
from matplotlib.pyplot import xlim
from numpy import stack

train_base=pd.read_parquet(TRAIN_DIR/"train_base.parquet")

#cast the datatypes to save memory
train_base["date_decision"]=pd.to_datetime(train_base["date_decision"])
train_base["case_id"]=pd.to_numeric(train_base["case_id"],downcast="integer")
train_base["MONTH"]=pd.to_numeric(train_base["MONTH"],downcast="integer")
train_base["WEEK_NUM"]=pd.to_numeric(train_base["WEEK_NUM"],downcast="integer")
train_base["target"]=pd.to_numeric(train_base["target"],downcast="integer")
train_base["date_decision"].drop_duplicates().sort_values().diff().sum().days
len(train_base["date_decision"].unique())
train_base["date_decision"].dt.date.max()
fig,axes=plt.subplots(2,2,figsize=(15,15))
plt.rc("figure",figsize=(15,15))
sns.histplot(train_base["target"],ax=axes[0,0]);
sns.histplot(train_base,x="date_decision",bins=30,hue="target",ax=axes[0,1]);
sns.histplot(train_base["MONTH"],ax=axes[1,0],bins=30);
sns.histplot(train_base["WEEK_NUM"],ax=axes[1,1]);
#axes[1,0].set_xticks([201901, 201902, 201903, 201904, 201905, 201906, 201907, 201908, 201909,
#       201910, 201911, 201912, 202001, 202002, 202003, 202004, 202005, 202006,
#       202007, 202008, 202009, 202010])
#axes[1,0].set_xticklabels([1, 2, 3, 4, 5, 6, 7, 8, 9,
#       10, 11, 12, 1, 2, 3, 4, 5, 6,
#       7, 8, 9, 10],rotation=180)
#axes[1,0].set_xlim([202001,202010])
#fig2=plt.figure()
#sns.histplot(train_base,x="date_decision",bins=30,hue="target");
#plt.tight_layout()
#train_base["MONTH"].value_counts().sort_values()


In [4]:
#check the proportion of null values in train files
total_nulls=[]
shapes=[]
for file in red.path:
    df=pl.read_parquet(file)
    total_nulls.append(df.null_count().to_pandas().sum().sum())
    shapes.append(np.prod(df.shape))
    del df
red["total_nulls"]=total_nulls
red["total_records"]=shapes
red["nulls_proportion"]=red["total_nulls"]/red["total_records"]
red.sort_values(by="nulls_proportion",ascending=False)

,size,path,filename,total_nulls,total_records,nulls_proportion
7,62M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_0.parquet,credit_bureau_a_1_0,244050042,324548748,0.751967
0,175M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_1.parquet,credit_bureau_a_1_1,308479374,474726168,0.649805
1,118M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_2.parquet,credit_bureau_a_1_2,187095066,295760990,0.632589
14,30M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_static_cb_0.parquet,static_cb_0,49375701,79525228,0.620881
6,69M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_1_3.parquet,credit_bureau_a_1_3,101862954,164266517,0.620108
30,1.2M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_debitcard_1.parquet,debitcard_1,450239,943812,0.477043
12,42M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_person_1.parquet,person_1,51051536,110037667,0.463946
26,5.6M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_2_0.parquet,credit_bureau_a_2_0,37184092,100624589,0.369533
21,9.6M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_2_1.parquet,credit_bureau_a_2_1,53702386,149374371,0.359515
15,30M,/home/xiong/Downloads/home-credit-credit-risk-model-stability/parquet_files/train/train_credit_bureau_a_2_2.parquet,credit_bureau_a_2_2,114892314,339977184,0.337941


# READ OTHER FILES

In [ ]:
#we checked some files, the files are large, we need polars to analyze
d0=pl.read_parquet(TRAIN_DIR/"train_credit_bureau_a_1_0.parquet")
d1=pl.read_parquet(TRAIN_DIR/"train_credit_bureau_a_1_1.parquet")
d2=pl.read_parquet(TRAIN_DIR/"train_credit_bureau_a_1_2.parquet")
d3=pl.read_parquet(TRAIN_DIR/"train_credit_bureau_a_1_3.parquet")
#d3.filter(pl.col("case_id")==51903)
d3["case_id"]

In [3]:
#functions to read files and join them into one table

#we need to handle below issues when join different tables into one
# 1.read each files and cast the datatypes(numeric,date) to save memory
# 2.for depth=[1,2],there are duplciated case_id with the different attributes in one table,we need to aggregate these data into one single record
# 3.concat vertically files with the same format and aggregate data if there are duplicated case_id
# 4.join horizontally all processed files together based on case_id and base data
# 5.drop the columns with large null values,and convert the string columns to catgory to save memory
# 6.keep columns with high correlation to target
# 7.care for the information leakage

#def read_file(path,depth=None):
#    df=pl.read_parquet(path)
#    df=df.pipe(handle_datatypes)
#    if depth in [1,2]:
#        df=df.group_by("case_id").agg(drop_duplicated_case(df))
#    return df


def read_files(path_exp,depth=None):
    chunks=[]
    for path in glob(str(path_exp)):
        df=pl.read_parquet(path)
        df=df.pipe(handle_datatypes)
        if depth in [1,2]:
            df=df.group_by("case_id").agg(drop_duplicated_case(df))
        chunks.append(df)
    df=pl.concat(chunks,how="vertical_relaxed")
    return df


def handle_datatypes(df :pl.DataFrame) ->pl.DataFrame:
    for col in df.columns:
        if col in ["case_id","MONTH"]:
            df=df.with_columns(pl.col(col).cast(pl.Int32))
        if col in ["WEEK_NUM","target"]:
            df=df.with_columns(pl.col(col).cast(pl.Int8))
        if col in ["date_decision"]:
            df=df.with_columns(pl.col(col).cast(pl.Date))
        if col in ["num_group1","num_group2"]:
            df=df.with_columns(pl.col(col).cast(pl.Int16))
        if col[-1] in ["P","A"]:
            df=df.with_columns(pl.col(col).cast(pl.Float64))
        if col[-1] in ["D"]:
            df=df.with_columns(pl.col(col).cast(pl.Date))
        
    return df
 
 
def drop_duplicated_case(df:pl.DataFrame)->pl.DataFrame:
    return [pl.col(col).max().alias(f"max_{col}") for col in df.columns if col not in["case_id","MONTH","WEEK_NUM","date_decision","target"]]


def drop_nas(df:pl.DataFrame)->pl.DataFrame:
    for col in df.columns:
        if col not in ["target", "case_id", "WEEK_NUM","MONTH","date_decision"]:
            null_cnt=df[col].is_null().mean()
            if (null_cnt>0.9) | ((df[col].dtype==pl.String) and (df[col].n_unique()==1)):
                df=df.drop(col)
    return df



def join_tbls(df_base,**kwargs):
    for i,df in kwargs.items():
        for col in df.columns:
            if col in ["num_group1","num_group2","max_num_group1","max_num_group2"]:
                df=df.drop(col)
        df_base=df_base.join(df,on="case_id",how="left",suffix=f"_{i}")
    return df_base


def drop_na_cols(df:pd.DataFrame)->pd.DataFrame:
    na_prop=(df.isna().sum())/len(df)
    na_cols=list(na_prop[na_prop>0.9].index)
    df=df.drop(columns=na_cols)
    return df

def drop_unique_cols(df:pd.DataFrame)->pd.DataFrame:
    unique_cols=[]
    for col in df.columns:
        if df[col].nunique()==1:
            unique_cols.append(col)
    df=df.drop(columns=unique_cols)
    return df

def convert_string_to_category(df:pd.DataFrame) ->pd.DataFrame:
    for col in df.columns: 
        if df[col].dtype in ["string","object"] and col[-1] in ["M","T","L"]:
            df[col]=df[col].astype("category")
    return df

def convert_date_to_datetime(df:pd.DataFrame) ->pd.DataFrame:
    for col in df.columns:
        if (col[-1] in ["D"]) or (col in ["date_decision"]):
            #df[col]=df[col].astype("datetime64")
            df[col]=pd.to_datetime(df[col],format='%Y-%m-%d')          
    return df

def handle_dates(df:pd.DataFrame)->pd.DataFrame:
    for col in df.columns:
        if col[-1] in ["D"]:
            df[col]=(df[col]-df["date_decision"]).dt.days
            df[col]=df[col].astype("float32")

    df["year"]=df["date_decision"].dt.year
    df["month"]=df["date_decision"].dt.month
    df["weekday"]=df["date_decision"].dt.weekday
    df=df.drop(["date_decision"],axis=1)


    return df
            

    


In [4]:
#train_applprev_1=pl.concat([pl.read_parquet(TRAIN_DIR/"train_applprev_1_0.parquet").pipe(handle_datatypes),
#          pl.read_parquet(TRAIN_DIR/"train_applprev_1_1.parquet").pipe(handle_datatypes)],          
#          how="vertical_relaxed")
#train_applprev_2=pl.read_parquet(TRAIN_DIR/"train_applprev_1_2.parquet").pipe(handle_datatypes)

train_base=pl.read_parquet(TRAIN_DIR/"train_base.parquet").pipe(handle_datatypes)
train_data_store={
                  "train_applprev_1":read_files(TRAIN_DIR/"train_applprev_1*",1),
                  "train_applprev_2":read_files(TRAIN_DIR/"train_applprev_2*",2),
                  "train_credit_bureau_a_1":read_files(TRAIN_DIR/"train_credit_bureau_a_1*",1),
                  "train_credit_bureau_a_2":read_files(TRAIN_DIR/"train_credit_bureau_a_2*",2),
                  "train_credit_bureau_b_1":read_files(TRAIN_DIR/"train_credit_bureau_b_1*",1),
                  "train_credit_bureau_b_2":read_files(TRAIN_DIR/"train_credit_bureau_b_2*",1),
                  "train_static":read_files(TRAIN_DIR/"train_static_0*"),
                  "train_static_cb":read_files(TRAIN_DIR/"train_static_cb*"),
                  "train_debitcard_1":read_files(TRAIN_DIR/"train_debitcard_1*",1),
                  "train_deposit_1":read_files(TRAIN_DIR/"train_deposit_1*",1),
                  "train_other_1":read_files(TRAIN_DIR/"train_other_1*",1),
                  "train_person_1":read_files(TRAIN_DIR/"train_person_1*",1),
                  "train_tax_registry_a_1":read_files(TRAIN_DIR/"train_tax_registry_a_1*",1),
                  "train_tax_registry_b_1":read_files(TRAIN_DIR/"train_tax_registry_b_1*",1),
                  "train_tax_registry_c_1":read_files(TRAIN_DIR/"train_tax_registry_c_1*",1),
                  "train_person_2":pl.read_parquet(TRAIN_DIR/"train_person_2.parquet").pipe(handle_datatypes).filter(pl.col("num_group1")==0).filter(pl.col("num_group2")==0)
}

#train_person_2=read_files(TRAIN_DIR/"train_person_2*",2)

train_df=join_tbls(train_base,**train_data_store)


In [5]:
test_base=pl.read_parquet(TEST_DIR/"test_base.parquet").pipe(handle_datatypes)
test_data_store={
                  "test_applprev_1":read_files(TEST_DIR/"test_applprev_1*",1),
                  "test_applprev_2":read_files(TEST_DIR/"test_applprev_2*",2),
                  "test_credit_bureau_a_1":read_files(TEST_DIR/"test_credit_bureau_a_1*",1),
                  "test_credit_bureau_a_2":read_files(TEST_DIR/"test_credit_bureau_a_2*",2),
                  "test_credit_bureau_b_1":read_files(TEST_DIR/"test_credit_bureau_b_1*",1),
                  "test_credit_bureau_b_2":read_files(TEST_DIR/"test_credit_bureau_b_2*",1),
                  "test_static":read_files(TEST_DIR/"test_static_0*"),
                  "test_static_cb":read_files(TEST_DIR/"test_static_cb*"),
                  "test_debitcard_1":read_files(TEST_DIR/"test_debitcard_1*",1),
                  "test_deposit_1":read_files(TEST_DIR/"test_deposit_1*",1),
                  "test_other_1":read_files(TEST_DIR/"test_other_1*",1),
                  "test_person_1":read_files(TEST_DIR/"test_person_1*",1),
                  "test_tax_registry_a_1":read_files(TEST_DIR/"test_tax_registry_a_1*",1),
                  "test_tax_registry_b_1":read_files(TEST_DIR/"test_tax_registry_b_1*",1),
                  "test_tax_registry_c_1":read_files(TEST_DIR/"test_tax_registry_c_1*",1),
                  "test_person_2":pl.read_parquet(TEST_DIR/"test_person_2.parquet").pipe(handle_datatypes).filter(pl.col("num_group1")==0).filter(pl.col("num_group2")==0)
}

#test_person_2=read_files(test_DIR/"test_person_2*",2)

test_df=join_tbls(test_base,**test_data_store)

In [ ]:

#train=[train_applprev_1,train_applprev_2,train_credit_bureau_a_1,train_credit_bureau_a_2,train_credit_bureau_b_1,train_credit_bureau_b_2,train_static,train_static_cb,train_debitcard_1,train_deposit_1,train_other_1,
#       train_person_1, train_person_2,train_tax_registry_a_1,train_tax_registry_b_1,train_tax_registry_c_1]

#find the below cols belongs to which table
#for i,df in enumerate(train):
#    cols=df.columns
#    for col in cols:
#        if col in ["max_relatedpersons_role_762T","max_empls_economicalst_849M","max_empls_employedfrom_796D","max_empls_employer_name_740M"]:
#            print(i)

#train_list=[train_applprev_1,train_applprev_2,train_credit_bureau_a_1,train_credit_bureau_a_2,train_credit_bureau_b_1,train_credit_bureau_b_2,train_static,train_static_cb,train_debitcard_1,train_deposit_1,train_other_1,
#       train_person_1, train_person_2_,train_tax_registry_a_1,train_tax_registry_b_1,train_tax_registry_c_1]

#for i,df in enumerate(train_list):
#    for col in df.columns:
#        if col in ["num_group1","num_group2","max_num_group1","max_num_group2"]:
#            df=df.drop(col)
#    train_base=train_base.join(df,on="case_id",how="left",suffix=f"_{i}")


In [6]:

#free memory
del train_data_store
del test_data_store
gc.collect()


4

In [7]:
train_df=drop_nas(train_df)
test_df=test_df.select([col for col in train_df.columns if col!="target"])


In [8]:
train_df=train_df.to_pandas()
test_df=test_df.to_pandas()

In [9]:

# convert object to category
train_df=convert_string_to_category(train_df)
test_df=convert_string_to_category(test_df)

In [10]:
#convert date to datetime
train_df=convert_date_to_datetime(train_df)
test_df=convert_date_to_datetime(test_df)

In [11]:
#convert dates to float
train_df=handle_dates(train_df)
test_df=handle_dates(test_df)

In [12]:
#drop columns with only one value
train_df=drop_unique_cols(train_df)



In [13]:
#drop na columns
train_df=drop_na_cols(train_df)

In [14]:
test_df=test_df[[col for col in train_df.columns if col !="target"]]

In [28]:

train_df.info(memory_usage="deep")
#train_df.memory_usage(deep=True).sort_values(ascending=False)
test_df.info(memory_usage="deep")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1526659 entries, 0 to 1526658
Columns: 342 entries, case_id to weekday
dtypes: bool(1), category(81), float32(39), float64(214), int32(5), int8(2)
memory usage: 2.9 GB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Columns: 341 entries, case_id to weekday
dtypes: bool(2), category(80), float32(39), float64(214), int32(5), int8(1)
memory usage: 36.1 KB


In [15]:
X=train_df.drop(["case_id","MONTH","WEEK_NUM","target"],axis=1)
y=train_df["target"]
weeks=train_df["WEEK_NUM"]

X_test=test_df[[col for col in X.columns]]


In [16]:
X_test["opencred_647L"]=X_test["opencred_647L"].astype("category")

/tmp/ipykernel_44578/324200725.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test["opencred_647L"]=X_test["opencred_647L"].astype("category")


In [17]:
X=X[:800000]
y=y[:800000]
weeks=weeks[:800000]

In [18]:
del train_df
gc.collect()

0

# cross split the data and train model based on each split, save the model fitted 

In [33]:
cv=StratifiedGroupKFold(n_splits=5,shuffle=False)

#fitted_models = []

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "max_depth": 8,
    "learning_rate": 0.05,
    "n_estimators": 1000,
    "colsample_bytree": 0.8, 
    "colsample_bynode": 0.8,
    "verbose": -1,
    "random_state": 42,
}
i=0
for train_idx,val_idx in cv.split(X,y,groups=weeks):
    i+=1
    train_X,train_y=X.iloc[train_idx],y.iloc[train_idx]
    valid_X,valid_y=X.iloc[val_idx],y.iloc[val_idx]
    model=lgb.LGBMClassifier(**params)
    model.fit(train_X,train_y,eval_set=[(valid_X,valid_y)],callbacks=[lgb.log_evaluation(100),lgb.early_stopping(100)])
    pd.to_pickle(model,f"/home/xiong/Downloads/model_{i}.pkl")


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.80692
[200]	valid_0's auc: 0.81003
Early stopping, best iteration is:
[183]	valid_0's auc: 0.810329
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.809665
[200]	valid_0's auc: 0.813896
[300]	valid_0's auc: 0.813717
Early stopping, best iteration is:
[218]	valid_0's auc: 0.814023
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.8134
[200]	valid_0's auc: 0.816991
Early stopping, best iteration is:
[188]	valid_0's auc: 0.817264
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.80268
[200]	valid_0's auc: 0.806874
[300]	valid_0's auc: 0.806597
Early stopping, best iteration is:
[229]	valid_0's auc: 0.807127
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.8152
[200]	valid_0's auc: 0.818631
[300]	valid_0's auc: 0.818837
Early stopping, best iteration is:
[252]	valid_

# load the models fitted, and predict the target probability for each model,get the mean probability for each target

In [20]:
ls_models=glob("/home/xiong/Downloads/*.pkl")
models=[pd.read_pickle(model) for model in ls_models]

y_preds=[model.predict_proba(X_test) for model in models]
#y_preds=[model.predict(X_test) for model in models]

y_preds_f=np.mean(y_preds,axis=0)
result=y_preds_f[:,1]

pd.DataFrame({"case_id":test_df["case_id"].to_numpy(),"score":result}).to_csv("/home/xiong/Downloads/result_1.csv",index=False)


# use pipeline to integrate the data preprocessing, encode the categorical data with one-hot encoder and then train the data


In [53]:
from sklearn import pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cv=StratifiedGroupKFold(n_splits=5,shuffle=False)

#fitted_models = []

params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "max_depth": 8,
    "learning_rate": 0.05,
    "n_estimators": 1000,
    "colsample_bytree": 0.8, 
    "colsample_bynode": 0.8,
    "verbose": -1,
    "random_state": 42,
}

y_proba=[]
cat_cols=[c for c in X.columns if X[c].dtype=="category"]


for train_idx,val_idx in cv.split(X,y,groups=weeks):
    train_X,train_y=X.iloc[train_idx],y.iloc[train_idx]
    valid_X,valid_y=X.iloc[val_idx],y.iloc[val_idx]
    model=lgb.LGBMClassifier(**params)
    col_trans=ColumnTransformer([('cat',OneHotEncoder(handle_unknown="ignore"),cat_cols)])
    my_pipe=pipeline.Pipeline(steps=[
        ('onehot',col_trans),
        ('lgbm',model),
        ])
    

    #somehow the lgbm fit parameters can not be recognized by the fit method, and I need to transform the valid_X with one hot encoder before put it into the fit eval_set param
    #here i decide to apply the pipeline method separately

    #my_pipe.fit(train_X,train_y,lgbm__eval_set=[(valid_X,valid_y)],lgbm__callbacks=[lgb.log_evaluation(100),lgb.early_stopping(100)])

    #step 1: fit and transform the data with one hot encoder
    train_X_eval=my_pipe["onehot"].fit_transform(train_X)
    valid_X_eval=my_pipe["onehot"].transform(valid_X)
    #step 2:fit the lgbm estimator on the transformed data
    my_pipe.steps[-1][1].fit(train_X_eval,train_y,eval_set=[(valid_X_eval,valid_y)],callbacks=[lgb.log_evaluation(100),lgb.early_stopping(100)])

    #
    y_proba.append(my_pipe.predict_proba(X_test))


    

/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.731022
[200]	valid_0's auc: 0.736129
[300]	valid_0's auc: 0.738078
[400]	valid_0's auc: 0.739033
[500]	valid_0's auc: 0.739604
[600]	valid_0's auc: 0.739808
[700]	valid_0's auc: 0.739841
Early stopping, best iteration is:
[665]	valid_0's auc: 0.73995


/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.723474
[200]	valid_0's auc: 0.729198
[300]	valid_0's auc: 0.73116
[400]	valid_0's auc: 0.731966
[500]	valid_0's auc: 0.732232
[600]	valid_0's auc: 0.73213
Early stopping, best iteration is:
[500]	valid_0's auc: 0.732232


/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.731328
[200]	valid_0's auc: 0.737402
[300]	valid_0's auc: 0.739731
[400]	valid_0's auc: 0.740503
[500]	valid_0's auc: 0.741184
[600]	valid_0's auc: 0.741578
[700]	valid_0's auc: 0.741647
[800]	valid_0's auc: 0.741909
[900]	valid_0's auc: 0.741802
Early stopping, best iteration is:
[807]	valid_0's auc: 0.741958


/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.730422
[200]	valid_0's auc: 0.73541
[300]	valid_0's auc: 0.736613
[400]	valid_0's auc: 0.737026
[500]	valid_0's auc: 0.737155
[600]	valid_0's auc: 0.736855
Early stopping, best iteration is:
[519]	valid_0's auc: 0.737253


/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/xiong/.local/lib/python3.12/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


In [55]:
np.mean(y_proba,axis=0)[:,1]

array([0.01021688, 0.01190751, 0.00714107, 0.0256703 , 0.01192885,
       0.01557719, 0.05056834, 0.02297124, 0.02061368, 0.02559971])